### **Task 1: Conceptual Understanding**

**What is the difference between "Love" and "love" in NLP?**
---

According to NLP model "Love" and "love" are distinct entities, even though both represent same semantic meaning. If the model treats them as different tokens then we unnecessarily increase the vocabulary size, so we apply Normalization before training to avoid redundancy.

**What happens if stopwords are not removed?**
---

Stopwords are high-frequency words like "the, is, in, at", if these are left in the dataset without careful management, several issues arise:

-Stopwords often carry very little unique meaning, they can drown out the content words that actually define the topic of a document.

-Because they appear so frequently, they occupy a massive portion of data which leads to longer training times and higher memory usage.


**Provide two real-world scenarios where removing stopwords can be harmful.**
---

Removing stopwords is standard practice, but doing it blindly can cause issues.

Let's consider the sentence: "The movie was not good."

If the preprocessing model removes "not" as a stopword, the model sees "movie good" and predicts a positive sentiment. In negation detection, stopwords are the most important words in the sentence, so removing "to, for, it" results in broken speech. For example "To be or not to be" would be reduced to "be be," losing the entire grammatical structure of the phrase.

**Explain the difference between stemming and lemmatization.**
---

Both techniques aim to reduce a word to its base form, but they use very different methods.

Stemming:

-Stemming is a rule-based approach that chops off the ends of words, it doesn't care about linguistics instead just follows an algorithm.

-It strips suffixes like "-ing, -ed, -ies".

-It often results in non-words, like "studies" and "studying" both become "studi".

-It is very fast and efficient but lacks precision.

Lemmatization:

-Lemmatization is a vocabulary approach, where it aims to return the "Lemma" —the dictionary base form of a word.

-It looks at the Part of Speech (POS) tag of the word, like it knows that "better" is a form of "good".

-It always results in a valid word. "Studies" becomes "study".

-It is much more accurate and preserves semantic meaning, but it is computationally more expensive because it requires a dictionary lookup and context analysis.

### Task 2:Build Advanced Preprocessing Function

In [34]:
import re

def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    # Convert text to lowercase
    text=text.lower()

    # Remove URLs and email-like patterns
    text=re.sub(r'https?://\S+|www\.\S+', '', text) # URLs
    text=re.sub(r'\S+@\S+', '', text)              # Emails

    # Remove numbers
    text=re.sub(r'\d+', '', text)
    text=re.sub(r'[^\w\s]', '', text)

    # Handle repeated characters
    text=re.sub(r'(.)\1{2,}', r'\1\1', text)
    tokens=text.split()

    # Remove very short tokens (length <= 2), keeping meaningful exceptions
    meaningful_short_words={'i', 'a', 'is', 'am', 'do', 'go', 'up', 'no', 'he', 'we', 'me','us', 'my', 'to', 'in', 'on', 'it', 'so', 'as', 'be', 'an', 'of', 'at', 'or', 'if', 'ok'}

    filtered_tokens=[]
    for token in tokens:
        if len(token)>2 or token in meaningful_short_words:
            if token=='soo':
                token='so'
            filtered_tokens.append(token)

    # Remove extra spaces
    text = ' '.join(filtered_tokens)
    return text

In [43]:
test_strings=[
    "I have 2 dogs",
    "This    is   good",
    "soooo goooood!!!",
    "WOW!!! This is GREAT!!!",
    "Visit http://example.com now and email me@test.com"
]

for s in test_strings:
    print(f"Original: {s}")
    print(f"Cleaned:  {preprocess_text(s)}\n")

Original: I have 2 dogs
Cleaned:  i have dogs

Original: This    is   good
Cleaned:  this is good

Original: soooo goooood!!!
Cleaned:  so good

Original: WOW!!! This is GREAT!!!
Cleaned:  wow this is great

Original: Visit http://example.com now and email me@test.com
Cleaned:  visit now and email



### Task 3: Stress Testing

In [47]:
sample_strings=[
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]
for sa in sample_strings:
    print(f"Original: {sa}")
    print(f"Cleaned:  {preprocess_text(sa)}\n")


Original: Get 100% FREE access now!!!
Cleaned:  get free access now

Original: I absolutely looooved this product 😍😍
Cleaned:  i absolutely looved this product

Original: Worst service ever... 0/10
Cleaned:  worst service ever

Original: Call me at 9876543210
Cleaned:  call me at

Original: This is THE best course!!!
Cleaned:  this is the best course

Original: Visit https://openai.com now!
Cleaned:  visit now

Original: Nooooo this is baaad!!!
Cleaned:  noo this is baad

Original: OK OK OK I got it
Cleaned:  ok ok ok i got it

Original: Win $$$ now!!! Limited offer!!!
Cleaned:  win now limited offer

Original: I am not happy with this
Cleaned:  i am not happy with this



### Task 4: Token Analytics

In [50]:
def analyze_tokens(original, cleaned):
    tokens=cleaned.split()
    total_tokens=len(tokens)
    unique_tokens=len(set(tokens))
    avg_length=sum(len(t) for t in tokens)/total_tokens if total_tokens>0 else 0

    noise_score = ((len(original)-len(cleaned))/len(original))*100 if len(original)>0 else 0

    return {
        "total": total_tokens,
        "unique": unique_tokens,
        "avg_len": round(avg_length, 2),
        "noise": round(noise_score, 2)
    }
results = []
for sa in sample_strings:
    cleaned=preprocess_text(sa)
    stats=analyze_tokens(sa, cleaned)
    results.append({"original": sa, "cleaned": cleaned, "stats": stats})

    print(f"Original: {sa}")
    print(f"Cleaned:  {cleaned}")
    print(f"Stats:    Tokens: {stats['total']} | Unique: {stats['unique']} | Avg Len: {stats['avg_len']}\n")

most_noisy=max(results, key=lambda x: x['stats']['noise'])
most_meaningful=max(results, key=lambda x: x['stats']['total'])

print(f"Most noisy sentence: '{most_noisy['original']}' ({most_noisy['stats']['noise']}% reduced)")
print(f"Most meaningful tokens retained'{most_meaningful['original']}' ({most_meaningful['stats']['total']} tokens retained)")

Original: Get 100% FREE access now!!!
Cleaned:  get free access now
Stats:    Tokens: 4 | Unique: 4 | Avg Len: 4.0

Original: I absolutely looooved this product 😍😍
Cleaned:  i absolutely looved this product
Stats:    Tokens: 5 | Unique: 5 | Avg Len: 5.6

Original: Worst service ever... 0/10
Cleaned:  worst service ever
Stats:    Tokens: 3 | Unique: 3 | Avg Len: 5.33

Original: Call me at 9876543210
Cleaned:  call me at
Stats:    Tokens: 3 | Unique: 3 | Avg Len: 2.67

Original: This is THE best course!!!
Cleaned:  this is the best course
Stats:    Tokens: 5 | Unique: 5 | Avg Len: 3.8

Original: Visit https://openai.com now!
Cleaned:  visit now
Stats:    Tokens: 2 | Unique: 2 | Avg Len: 4.0

Original: Nooooo this is baaad!!!
Cleaned:  noo this is baad
Stats:    Tokens: 4 | Unique: 4 | Avg Len: 3.25

Original: OK OK OK I got it
Cleaned:  ok ok ok i got it
Stats:    Tokens: 6 | Unique: 4 | Avg Len: 2.0

Original: Win $$$ now!!! Limited offer!!!
Cleaned:  win now limited offer
Stats:    Tok

### Task 5: Frequency Analysis


In [51]:
from collections import Counter
all_tokens=[]
for item in results:
    all_tokens.extend(item['cleaned'].split())
word_counts=Counter(all_tokens)
top_10=word_counts.most_common(10)
least_5=sorted(word_counts.items(), key=lambda x: x[1])[:5]

print("Top 10 frequent")
for word, count in top_10:
    print(f"{word}: {count}")

print("\nLeast 5 frequent")
for word, count in least_5:
    print(f"{word}: {count}")

Top 10 frequent
this: 4
now: 3
i: 3
ok: 3
is: 2
get: 1
free: 1
access: 1
absolutely: 1
looved: 1

Least 5 frequent
get: 1
free: 1
access: 1
absolutely: 1
looved: 1


### Task 6: Build Full Pipeline

In [60]:
def full_pipeline(text_list):
    all_tokens = []
    clean_sentences = []

    for text in text_list:
        cleaned_text = preprocess_text(text)
        clean_sentences.append(cleaned_text)
        all_tokens.extend(cleaned_text.split())

    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }

pipeline_results = full_pipeline(sample_strings)
print("\nAll Tokens:")
print(pipeline_results['tokens'])
print("\nCleaned Sentences:")
for sentence in pipeline_results['clean_sentences']:
    print(f"- {sentence}")


All Tokens:
['get', 'free', 'access', 'now', 'i', 'absolutely', 'looved', 'this', 'product', 'worst', 'service', 'ever', 'call', 'me', 'at', 'this', 'is', 'the', 'best', 'course', 'visit', 'now', 'noo', 'this', 'is', 'baad', 'ok', 'ok', 'ok', 'i', 'got', 'it', 'win', 'now', 'limited', 'offer', 'i', 'am', 'not', 'happy', 'with', 'this']

Cleaned Sentences:
- get free access now
- i absolutely looved this product
- worst service ever
- call me at
- this is the best course
- visit now
- noo this is baad
- ok ok ok i got it
- win now limited offer
- i am not happy with this


### Task 7: Error Handling

In [65]:
error_handling_test_strings = [
    "",
    "😊😂👍",
    "1234567890",
    "   ",
    "Mix of 123 and emojis 😂"
]

print("Error Handling Test Cases: ")
for s in error_handling_test_strings:
    cleaned_s=preprocess_text(s)
    print(f"Original: '{s}' (Type: {type(s)})\nCleaned:  '{cleaned_s}' (Length: {len(cleaned_s)})\n")


Error Handling Test Cases: 
Original: '' (Type: <class 'str'>)
Cleaned:  '' (Length: 0)

Original: '😊😂👍' (Type: <class 'str'>)
Cleaned:  '' (Length: 0)

Original: '1234567890' (Type: <class 'str'>)
Cleaned:  '' (Length: 0)

Original: '   ' (Type: <class 'str'>)
Cleaned:  '' (Length: 0)

Original: 'Mix of 123 and emojis 😂' (Type: <class 'str'>)
Cleaned:  'mix of and emojis' (Length: 17)

